In [ ]:
from datetime import timedelta
import pandas as pd

def map_dates_with_offset(base_dates, dates, offset=0, offset_unit='days'):
    """
    Map each date in base_dates to the max date in dates that is <= (base_date - offset).
    
    Parameters:
        base_dates (list of str or datetime): The base reference dates.
        dates (list of str or datetime): The dates to search in.
        offset (int): The number of units to subtract from each base date.
        offset_unit (str): The unit of offset. Can be 'days', 'months', or 'years'.
    
    Returns:
        dict: Mapping from base_date to the maximum eligible date from dates.
    """
    # Convert to pandas datetime Series
    base_dates = pd.to_datetime(base_dates)
    dates = pd.to_datetime(dates)
    
    # Preprocess based on offset
    if offset_unit == 'days':
        offset_delta = pd.to_timedelta(offset, unit='D')
        adjusted_base_dates = base_dates - offset_delta
    elif offset_unit == 'months':
        adjusted_base_dates = base_dates - pd.DateOffset(months=offset)
    elif offset_unit == 'years':
        adjusted_base_dates = base_dates - pd.DateOffset(years=offset)
    else:
        raise ValueError("Unsupported offset_unit. Use 'days', 'months', or 'years'.")

    # Sort dates for efficient max lookup
    dates = dates.sort_values()

    result = {}
    for bdate, adj_bdate in zip(base_dates, adjusted_base_dates):
        eligible_dates = dates[dates <= adj_bdate]
        result[bdate] = eligible_dates.max() if not eligible_dates.empty else None

    return result


In [ ]:
base_dates = ['2023-05-01', '2023-06-15', '2023-08-01']
dates = ['2023-01-01', '2023-04-01', '2023-05-10', '2023-06-10', '2023-07-01']

result = map_dates_with_offset(base_dates, dates, offset=30, offset_unit='days')
for k, v in result.items():
    print(f"{k.date()} → {v.date() if pd.notna(v) else None}")
